**HyDE RAG Implementation**

Some Important Specifications -
This notebook implements a basic Naive RAG (Retrieval-Augmented Generation) pipeline using:

1. Supabase as vector database
2. PyMuPDF for PDF reading
3. Sentence Transformers embeddings
4. BAAI/bge-small-en-v1.5 embedding model
5. vecs as pgvector wrapper
6. Groq for LLM inference

**WorkFlow**

User Query -> Tokenizer -> Generator LLM -> Hypothetical Document -> Embedding Model -> Dense Vector -> ANN Search in Vector DB -> Top-K Retrived Docs -> (Optional Re-Ranker) -> Final Prompt Construction -> Answer LLM

1. When a query arrives, it is in simple string format.
2. Then the string is converted to tokens
GPT → BPE
T5/LLaMA → SentencePiece
BERT → WordPiece
3. Then the tokened query is given to Gen LLM, to make hypothetical document which would use self attention layers to generate the next words to the previous ones based on probabilities.
4. Then the generated document is turned to embeddings

**NOTE: HyDE used a generation model and then an embedding model
Those embeddings are stored to vector database.**

5. Now documents similar to query are searched in vector DB through ANN Search (Approximate Nearest Neighbour) and similarity checked though cosine similarity

#Step 1: Install Libraries

In [ ]:
# Install dependencies
!pip install -q langchain langchain-community langchain-core
!pip install -q langchain-groq
!pip install -q sentence-transformers
!pip install -q supabase
!pip install -q psycopg2-binary
!pip install -q pypdf

In [ ]:
import os
from typing import List

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document

from sentence_transformers import SentenceTransformer

from supabase import create_client, Client

from langchain_groq import ChatGroq

#Step 2: Supabase, Embedding Model and LLM Setup

In [ ]:
# GROQ API KEY
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

# SUPABASE
SUPABASE_URL = "YOUR_SUPABASE_URL"
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
llm = ChatGroq(
    model_name="llama3-70b-8192",
    temperature=0
)

In [ ]:
#Create Table In Supabase
"""
RUN THIS SQL INSIDE SUPABASE SQL EDITOR FIRST

-- Enable pgvector
create extension if not exists vector;

-- Create table
create table documents (
  id bigserial primary key,
  content text,
  metadata jsonb,
  embedding vector(384)
);

-- Similarity search function
create or replace function match_documents (
  query_embedding vector(384),
  match_count int default 5
)
returns table (
  id bigint,
  content text,
  metadata jsonb,
  similarity float
)
language sql stable
as $$
  select
    id,
    content,
    metadata,
    1 - (documents.embedding <=> query_embedding) as similarity
  from documents
  order by documents.embedding <=> query_embedding
  limit match_count;
$$;
"""


#Step 3: PDF Loading

In [ ]:
pdf_path = "/content/sample.pdf"   # Upload PDF in Colab first

loader = PyPDFLoader(pdf_path)
pages = loader.load()

#Step 4: Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(pages)

print(f"Total chunks: {len(docs)}")

#Step 5: Storage in Supabase

In [ ]:
def embed_text(text: str):
    return embedding_model.encode(text).tolist()

for doc in docs:

    embedding = embed_text(doc.page_content)

    data = {
        "content": doc.page_content,
        "metadata": doc.metadata,
        "embedding": embedding
    }

    supabase.table("documents").insert(data).execute()

print("Documents stored successfully!")

#Step 6: HyDE Document Generation

In [ ]:
"""
HYDE = Hypothetical Document Embeddings

Instead of embedding the user query directly:
1. Generate a hypothetical answer/document
2. Embed THAT generated text
3. Use it for retrieval

This improves semantic retrieval significantly.
"""

def generate_hypothetical_document(query: str):

    prompt = f"""
    Write a detailed hypothetical answer/document for the question below.

    Question:
    {query}

    Hypothetical Answer:
    """

    response = llm.invoke(prompt)

    return response.content

#Step 7: Retriver

In [ ]:
def retrieve_documents(query: str, top_k=5):

    # STEP A: Generate HYDE document
    hyde_doc = generate_hypothetical_document(query)

    print("\n===== HYDE DOCUMENT =====\n")
    print(hyde_doc[:1000])

    # STEP B: Embed HYDE document
    query_embedding = embed_text(hyde_doc)

    # STEP C: Search Supabase
    result = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_count": top_k
        }
    ).execute()

    return result.data

#Step 8: Final Function Call

In [ ]:
def hyde_rag(query: str):

    retrieved_docs = retrieve_documents(query)

    context = "\n\n".join([
        doc["content"] for doc in retrieved_docs
    ])

    final_prompt = f"""
    Answer the question using ONLY the context below.

    CONTEXT:
    {context}

    QUESTION:
    {query}

    ANSWER:
    """

    response = llm.invoke(final_prompt)

    return response.content

#Step 9: Testing

In [ ]:
query = "What are the main objectives discussed in the document?"

answer = hyde_rag(query)

print("\n================ FINAL ANSWER ================\n")
print(answer)